In [2]:
from os import listdir
from os.path import isdir, isfile, join

import networkx as nx
import pandas as pd
import torch
import torch.multiprocessing

from helpers.slurm_tools import *
from helpers.tools import *

torch.multiprocessing.set_sharing_strategy('file_system')
from os import listdir
from os.path import isfile, join

In [57]:
mypath = "/home/stein_test/Causal_pretraining/multirun/experiment_1_std/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]

In [58]:
onlyfiles = [x for x in onlyfiles if "summary" in x]

In [59]:
stack = []
for x in onlyfiles:
    stack.append(pd.read_csv(x, index_col=0))

In [60]:
stack[0]

,5,1,8,9,3,6,7,4,2,0,...,5.29,1.29,8.29,9.29,3.29,6.29,7.29,4.29,2.29,0.29
HP,,,,,,,,,,,,,,,,,,,,,
seed,43,50,45,51,42,48,46,44,49,47,...,51,46,45,44,48,47,49,50,43,42
hparams_search,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,...,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml
model.optimizer_lr,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
data.batch_size,16,16,16,16,16,16,16,16,16,16,...,16,16,16,16,16,16,16,16,16,16
model.weight_decay,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,...,0,0,0,0,0,0,0,0,0,0
model.model_type,transformer,transformer,transformer,transformer,transformer,transformer,transformer,transformer,transformer,transformer,...,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp
model.corr_regularization,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
corr_input,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
regression_head,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [32]:
joint= pd.concat(stack, axis=1).T[["data.ds_name", "model.model_type", "Val_Auroc", "Val_MSE"]]
joint["Val_Auroc"] = joint["Val_Auroc"].astype(float)
joint["Val_MSE"] = joint["Val_MSE"].astype(float)

joint.groupby(["data.ds_name", "model.model_type"]).mean()

HP                                Val_Auroc   Val_MSE
data.ds_name    model.model_type                     
base            bidirectional      0.999723  0.003645
                convM              0.999492  0.005584
                mlp                0.999588  0.004667
                transformer        0.999605  0.004235
                unidirectional     0.999746  0.003680
big_base_mirror bidirectional      0.616386  0.071914
                convM              0.506116  0.082694
                mlp                0.501286  0.082275
                transformer        0.604423  0.076602
                unidirectional     0.603570  0.079572
big_nl          bidirectional      0.709001  0.065371
                convM              0.504203  0.079942
                mlp                0.498411  0.080265
                transformer        0.512214  0.080731
                unidirectional     0.657722  0.072013
big_poly        bidirectional      0.539421  0.076636
                convM              0.514786  0.081701
                mlp                0.512487  0.078301
                transformer        0.534717  0.078187
                unidirectional     0.782321  0.057270
nl              bidirectional      0.907597  0.038839
                convM              0.903236  0.042587
                mlp                0.897637  0.044948
                transformer        0.905058  0.039926
                unidirectional     0.906535  0.040757
poly            bidirectional      0.925065  0.035742
                convM              0.922431  0.037287
                mlp                0.919533  0.039458
                transformer        0.924174  0.035993
                unidirectional     0.925728  0.034465

In [33]:
# Check some data: 

stats, broke = check_slurm_run(onlyfiles[2] + "/", verbose=0)
stats


,5,1,8,9,3,6,7,4,2,0
HP,,,,,,,,,,
seed,46,48,49,51,43,50,42,47,45,44
hparams_search,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml,std_runs_slurm.yaml
model.optimizer_lr,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
data.batch_size,16,16,16,16,16,16,16,16,16,16
model.weight_decay,0,0,0,0,0,0,0,0,0,0
model.model_type,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp,mlp
model.corr_regularization,False,False,False,False,False,False,False,False,False,False
corr_input,False,False,False,False,False,False,False,False,False,False
regression_head,False,False,False,False,False,False,False,False,False,False


## summary helpers

In [5]:
def sum_best_performance(onlyfiles, onlyfiles2):
    stack = []
    for n in range(len(onlyfiles)):
        if onlyfiles2:
            a = pd.read_csv(onlyfiles[n], index_col=0).T
            b = pd.read_csv(onlyfiles2[n], index_col=0).T
            data = pd.concat([a,b])
        else:
            data = pd.read_csv(onlyfiles[n], index_col=0).T
        data["Val_Auroc"] = data["Val_Auroc"].astype(float)
        data["Val_MSE"] = data["Val_MSE"].astype(float)
        data = data[["model.model_type", "Val_Auroc"]]
        data.rename(columns={"Val_Auroc": onlyfiles[n].split("/")[-1][:-12]}, inplace=True)
        stack.append(data.groupby("model.model_type").max())
    return pd.concat(stack, axis=1)

def sum_best_performance_of_hps(onlyfiles, onlyfiles2,relevant = [ "model.corr_regularization", "corr_input", "regression_head"] ):
    compare = []
    for n in range(len(onlyfiles)):
        if onlyfiles2:
            a = pd.read_csv(onlyfiles[n], index_col=0).T
            b = pd.read_csv(onlyfiles2[n], index_col=0).T
            data = pd.concat([a,b])
        else:
            data = pd.read_csv(onlyfiles[n], index_col=0).T
        data["Val_Auroc"] = data["Val_Auroc"].astype(float)
        data["Val_MSE"] = data["Val_MSE"].astype(float)
        stack = []
        for x in relevant:
            res = data[[x, "Val_Auroc"]].groupby(x).max() 
            res.rename(columns={"Val_Auroc": x}, inplace=True)
            res.index = [onlyfiles[n].split("/")[-1][:-12] + "_" + x for x in res.index ]
            stack.append(res)
        compare.append(pd.concat(stack,axis=1))
    return pd.concat(compare)

def get_best_runs(onlyfiles, onlyfiles2): 
    stack = []
    for n in range(len(onlyfiles)):
        if onlyfiles2:
            a = pd.read_csv(onlyfiles[n], index_col=0).T
            b = pd.read_csv(onlyfiles2[n], index_col=0).T
            data = pd.concat([a,b])
        else:
            data = pd.read_csv(onlyfiles[n], index_col=0).T  
        data["Val_Auroc"] = data["Val_Auroc"].astype(float)
        data["Val_MSE"] = data["Val_MSE"].astype(float)
        for x in data["model.model_type"].unique():
            relevant = data[data["model.model_type"] == x].copy()
            relevant.sort_values("Val_MSE", inplace=True)
            stack.append(relevant[:1])
    return pd.concat(stack).sort_values("model.model_type")

def display_std_runs(data, what= "maxi", test2=True):
    if test2:
        data["Test1_Auroc"] = data["Test1_Auroc"].astype(float)
        data["Test2_AUROC"] = data["Test2_AUROC"].astype(float)
        data = data[["Test1_Auroc", "Test2_AUROC","model.model_type", "data.ds_name" ,  "model"  ]]
        data.loc[data["data.ds_name"].isnull(), "data.ds_name"] = "base"
        if what == "maxi":
            return data[["Test1_Auroc", "Test2_AUROC","model.model_type", "data.ds_name" ,  "model"]].groupby(["model.model_type", "data.ds_name",  "model"]).max()
        if what == "mean":
            return data[["Test1_Auroc", "Test2_AUROC","model.model_type", "data.ds_name" ,  "model"]].groupby(["model.model_type", "data.ds_name",  "model"]).mean()
        if what == "std":
            return data[["Test1_Auroc", "Test2_AUROC","model.model_type", "data.ds_name",  "model" ]].groupby(["model.model_type", "data.ds_name",  "model"]).std()
    else:
        data["Test1_Auroc"] = data["Test1_Auroc"].astype(float)
        data = data[["Test1_Auroc","model.model_type", "data.ds_name",  "model" ]]
        data.loc[data["data.ds_name"].isnull(), "data.ds_name"] = "base"
        if what == "maxi":
            return data[["Test1_Auroc", "model.model_type", "data.ds_name",  "model" ]].groupby(["model.model_type", "data.ds_name",  "model"]).max()
        if what == "mean":
            return data[["Test1_Auroc","model.model_type", "data.ds_name",  "model" ]].groupby(["model.model_type", "data.ds_name",  "model"]).mean()
        if what == "std":
            return data[["Test1_Auroc", "model.model_type", "data.ds_name",  "model" ]].groupby(["model.model_type", "data.ds_name",  "model"]).std()

# Experiment 1 - Grid results summarized.

In [3]:
mypath = "multirun/experiment_1_grid_search/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles = sorted([x for x in onlyfiles if "summary" in x])

In [4]:
mypath = "multirun/experiment_1_grid_search_2/"
onlyfiles2 = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles2 = sorted([x for x in onlyfiles2 if "summary" in x])

In [7]:
get_best_runs(onlyfiles, onlyfiles2)

HP,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,hparams_search,Val_Auroc,Val_MSE,data.ds_name,model,n_vars,max_lags
453,0.001,64,0,bidirectional,False,True,False,joint_slurm.yaml,0.953577,0.033576,big_base_mirror,medium.yaml,5,3
402,0.001,16,0,bidirectional,False,True,False,joint_slurm.yaml,0.907943,0.038455,nl,NaN,NaN,NaN
385,0.001,16,0,bidirectional,False,True,False,joint_slurm.yaml,0.809684,0.054513,big_nl,medium.yaml,5,3
43,0.001,16,0.01,bidirectional,False,True,False,joint_slurm.yaml,0.999739,0.003834,base,NaN,NaN,NaN
147,0.0001,16,0.01,bidirectional,False,True,False,joint_slurm.yaml,0.925393,0.034813,poly,NaN,NaN,NaN
129,0.001,256,0.01,bidirectional,False,True,False,joint_slurm.yaml,0.793406,0.061978,big_poly,medium.yaml,5,3
54,0.001,16,0.01,convM,True,True,True,joint_slurm.yaml,0.908207,0.041512,nl,NaN,NaN,NaN
466,0.001,256,0.01,convM,False,True,True,joint_slurm.yaml,0.511027,0.079870,big_nl,medium.yaml,5,3
157,0.001,16,0,convM,False,True,True,joint_slurm.yaml,0.515679,0.081632,big_base_mirror,medium.yaml,5,3
62,0.001,16,0,convM,False,True,False,joint_slurm.yaml,0.924792,0.036665,poly,NaN,NaN,NaN


In [8]:
sum_best_performance(onlyfiles, onlyfiles2)

HP,base,big_base_mirror,big_nl,big_poly,nl,poly
model.model_type,,,,,,
bidirectional,0.999753,0.953577,0.809684,0.793406,0.908286,0.927615
convM,0.999693,0.524568,0.515436,0.541951,0.908207,0.926249
mlp,0.999627,0.515738,0.512032,0.521789,0.903739,0.922108
transformer,0.999663,0.897439,0.563333,0.594415,0.905645,0.939629
unidirectional,0.999797,0.965605,0.803874,0.825641,0.908993,0.970863


In [9]:
sum_best_performance_of_hps(onlyfiles, onlyfiles2)

HP,model.corr_regularization,corr_input,regression_head
base_False,0.999783,0.986512,0.999785
base_True,0.999797,0.999797,0.999797
big_base_mirror_False,0.964874,0.515738,0.965605
big_base_mirror_True,0.965605,0.965605,0.532108
big_nl_False,0.809684,0.514038,0.809684
big_nl_True,0.734774,0.809684,0.563333
big_poly_False,0.825641,0.524581,0.825641
big_poly_True,0.756100,0.825641,0.541951
nl_False,0.908286,0.879267,0.908154
nl_True,0.908993,0.908993,0.908993


# Experiment 2 - Grid search - medium

In [10]:
mypath = "multirun/experiment_2_grid_search_medium/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles = sorted([x for x in onlyfiles if "summary" in x])

In [11]:
get_best_runs(onlyfiles, onlyfiles2=False)

HP,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,hparams_search,model,max_lags,n_vars,data.ds_name,Val_Auroc,Val_MSE
4,0.001,16,0,bidirectional,False,True,False,slurm_exp2.yaml,medium.yaml,3,5,joint,0.928198,0.029236
62,0.001,16,0.01,convM,False,True,False,slurm_exp2.yaml,medium.yaml,3,5,joint,0.871583,0.036523
17,0.001,64,0,mlp,True,True,False,slurm_exp2.yaml,medium.yaml,3,5,joint,0.935664,0.026672
69,0.001,64,0.01,transformer,False,True,False,slurm_exp2.yaml,medium.yaml,3,5,joint,0.941540,0.024755
84,0.001,64,0,unidirectional,True,True,False,slurm_exp2.yaml,medium.yaml,3,5,joint,0.891559,0.034746


In [12]:
sum_best_performance_of_hps(onlyfiles, onlyfiles2=False)

HP,model.corr_regularization,corr_input,regression_head
13-23-36_False,0.941540,NaN,0.94154
13-23-36_True,0.938384,0.94154,NaN


# Experiment 2 -  Grid search - Big

In [13]:
mypath = "multirun/experiment_2_grid_search_big/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles = sorted([x for x in onlyfiles if "summary" in x])
res = pd.concat([pd.read_csv(x, index_col=0) for x in onlyfiles],axis=1).T

In [14]:
get_best_runs(onlyfiles, onlyfiles2=False)

HP,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,hparams_search,data.ds_name,max_lags,n_vars,model,Val_Auroc,Val_MSE
21,0.001,16,0.01,bidirectional,False,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.964593,0.014471
63,0.001,16,0.01,bidirectional,False,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.929818,0.029429
90,0.001,16,0.01,convM,True,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.865958,0.037415
102,0.001,16,0.01,convM,True,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.865665,0.037434
9,0.0001,64,0,mlp,True,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.789235,0.047096
84,0.001,64,0.01,transformer,False,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.968994,0.012700
6,0.001,64,0,transformer,False,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.968055,0.012669
0,0.001,64,0.01,unidirectional,True,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.966476,0.015678
75,0.001,64,0.01,unidirectional,False,True,False,slurm_exp2.yaml,joint,3,5,big.yaml,0.967558,0.014001


In [15]:
sum_best_performance(onlyfiles, onlyfiles2=False)

HP,06-11-11-11-414287,09-19-11-10-730168,10-19-11-52-079296
model.model_type,,,
bidirectional,0.964593,0.929818,NaN
convM,0.865958,0.866393,NaN
transformer,0.973140,0.971673,NaN
unidirectional,0.966476,0.967558,NaN
mlp,NaN,NaN,0.789235


In [16]:
sum_best_performance_of_hps(onlyfiles, onlyfiles2=False)

HP,model.corr_regularization,corr_input,regression_head
06-11-11-11-414287_False,0.973140,NaN,0.973140
06-11-11-11-414287_True,0.969463,0.973140,NaN
09-19-11-10-730168_False,0.969059,NaN,0.971673
09-19-11-10-730168_True,0.971673,0.971673,NaN
10-19-11-52-079296_False,0.777165,NaN,0.789235
10-19-11-52-079296_True,0.789235,0.789235,NaN


# Experiment 1 - Std Runs

In [17]:
data = pd.read_csv("multirun/experiment_1_std/summary.csv", index_col=0).T
res = display_std_runs(data)
res

HP                                            Test1_Auroc  Test2_AUROC
model.model_type data.ds_name    model                                
bidirectional    big_base_mirror medium.yaml     0.953971     0.971049
                 big_nl          medium.yaml     0.807985     0.858170
                 big_poly        medium.yaml     0.803827     0.851144
convM            big_base_mirror medium.yaml     0.518909     0.534162
                 big_nl          medium.yaml     0.514575     0.513029
                 big_poly        medium.yaml     0.524451     0.525794
mlp              big_base_mirror medium.yaml     0.506958     0.505405
                 big_nl          medium.yaml     0.509978     0.505632
                 big_poly        medium.yaml     0.528956     0.530116
transformer      big_base_mirror medium.yaml     0.879106     0.902991
                 big_nl          medium.yaml     0.583993     0.606152
                 big_poly        medium.yaml     0.549549     0.552634
unidirectional   big_base_mirror medium.yaml     0.693940     0.716576
                 big_nl          medium.yaml     0.806000     0.852697
                 big_poly        medium.yaml     0.832014     0.884602

In [19]:
sum(data["model.corr_regularization"] == "True") / len(data)

0.1

In [18]:
res = display_std_runs(data, what = "maxi")
res.to_csv("scores/slurm_exp1_max.csv")
res = display_std_runs(data, what = "mean")
res.to_csv("scores/slurm_exp1_mean.csv")
res = display_std_runs(data, what = "std")
res.to_csv("scores/slurm_exp1_std.csv")

# Experiment 2 Medium - Std Runs

In [20]:
data = pd.read_csv("multirun/experiment_2_std_medium/summary.csv", index_col=0).T
display_std_runs(data)

,,HP,Test1_Auroc,Test2_AUROC
model.model_type,data.ds_name,model,,
bidirectional,joint,medium.yaml,0.931927,0.948010
convM,joint,medium.yaml,0.873121,0.891351
mlp,joint,medium.yaml,0.951703,0.966073
transformer,joint,medium.yaml,0.955934,0.968061
unidirectional,joint,medium.yaml,0.919554,0.933273


In [21]:
sum(data["model.corr_regularization"] == "True") / len(data)

0.40816326530612246

In [22]:
res = display_std_runs(data, what = "maxi")
res.to_csv("scores/slurm_exp2_m_max.csv")
res = display_std_runs(data, what = "mean")
res.to_csv("scores/slurm_exp2_m_mean.csv")
res = display_std_runs(data, what = "std")
res.to_csv("scores/slurm_exp2_m_std.csv")

# Experiment 2 Big - Std Runs

In [23]:
data = pd.read_csv("multirun/experiment_2_std_big/summary.csv", index_col=0).T
display_std_runs(data, )

,,HP,Test1_Auroc,Test2_AUROC
model.model_type,data.ds_name,model,,
bidirectional,joint,big.yaml,0.967851,0.976821
convM,joint,big.yaml,0.871109,0.892276
mlp,joint,big.yaml,0.791368,0.798233
transformer,joint,big.yaml,0.971022,0.978486
unidirectional,joint,big.yaml,0.969732,0.978401


In [24]:
res = display_std_runs(data, what = "maxi")
res.to_csv("scores/slurm_exp2_b_max.csv")
res = display_std_runs(data, what = "mean")
res.to_csv("scores/slurm_exp2_b_mean.csv")
res = display_std_runs(data, what = "std")
res.to_csv("scores/slurm_exp2_b_std.csv")

# Experiment 3 - Scale up - Grid search

In [46]:
mypath = "multirun/experiment_3_scale_up/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles = sorted([x for x in onlyfiles if "summary" in x])
res = pd.concat([pd.read_csv(x, index_col=0) for x in onlyfiles],axis=1).T

In [59]:
# I am still missing the STD runs here.
mypath = "multirun/experiment_3_scale_up_deep/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
onlyfiles = sorted([x for x in onlyfiles if "summary" in x])
res = pd.concat([pd.read_csv(x, index_col=0) for x in onlyfiles],axis=1).T
pd.DataFrame(res.groupby(["data.ds_name", "model"]).max()["Val_Auroc"]).to_csv("scores/slurm_exp3_pre_max.csv")

# STD

In [70]:
data = pd.read_csv("multirun/experiment_3_scale_up_std/summary.csv", index_col=0).T
res = display_std_runs(data)

In [73]:
data[data["data.ds_name"] == "scale_up_2"]

HP,seed,hparams_search,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,data.ds_name,max_lags,n_vars,model,Test1_Auroc,Test1_MSE,Test2_AUROC,Test2_MSE
1.1,44,std_runs_slurm.yaml,0.001,64,0.01,unidirectional,False,True,False,scale_up_2,3,7,big.yaml,0.908557,0.05034545,0.921397,0.04363391
3.1,42,std_runs_slurm.yaml,0.001,64,0.01,unidirectional,False,True,False,scale_up_2,3,7,big.yaml,0.889303,0.053441484,0.903663,0.04739018
4.1,43,std_runs_slurm.yaml,0.001,64,0.01,unidirectional,False,True,False,scale_up_2,3,7,big.yaml,0.920174,0.047355648,0.936330,0.03943889
2.1,46,std_runs_slurm.yaml,0.001,64,0.01,unidirectional,False,True,False,scale_up_2,3,7,big.yaml,0.674299,0.079047,0.699713,0.07721062
0.1,45,std_runs_slurm.yaml,0.001,64,0.01,unidirectional,False,True,False,scale_up_2,3,7,big.yaml,0.909774,0.04934624,0.926161,0.042238854
1.2,42,std_runs_slurm.yaml,0.001,16,0.01,unidirectional,False,True,False,scale_up_2,3,7,medium.yaml,0.716983,0.07390574,0.740079,0.0714415
3.2,46,std_runs_slurm.yaml,0.001,16,0.01,unidirectional,False,True,False,scale_up_2,3,7,medium.yaml,0.503115,0.08125919,0.501819,0.081328906
4.2,43,std_runs_slurm.yaml,0.001,16,0.01,unidirectional,False,True,False,scale_up_2,3,7,medium.yaml,0.501973,0.08137964,0.509171,0.08131104
2.2,45,std_runs_slurm.yaml,0.001,16,0.01,unidirectional,False,True,False,scale_up_2,3,7,medium.yaml,0.702671,0.075531684,0.722473,0.07390732
0.2,44,std_runs_slurm.yaml,0.001,16,0.01,unidirectional,False,True,False,scale_up_2,3,7,medium.yaml,0.720955,0.07358736,0.747751,0.07056258


In [30]:
data

HP,seed,hparams_search,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,data.ds_name,max_lags,n_vars,model,Test1_Auroc,Test1_MSE,Test2_AUROC,Test2_MSE
1,42,std_runs_slurm.yaml,0.001,16,0.01,convM,False,True,False,big_base_mirror,3,5,small.yaml,0.522613,0.082307614,0.527030,0.08112592
3,44,std_runs_slurm.yaml,0.001,16,0.01,convM,False,True,False,big_base_mirror,3,5,small.yaml,0.508889,0.082269676,0.509824,0.08110255
4,43,std_runs_slurm.yaml,0.001,16,0.01,convM,False,True,False,big_base_mirror,3,5,small.yaml,0.514402,0.082271524,0.525610,0.081070274
2,45,std_runs_slurm.yaml,0.001,16,0.01,convM,False,True,False,big_base_mirror,3,5,small.yaml,0.502777,0.08231586,0.505849,0.08116848
0,46,std_runs_slurm.yaml,0.001,16,0.01,convM,False,True,False,big_base_mirror,3,5,small.yaml,0.504894,0.08236103,0.512374,0.081128284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1.35,46,std_runs_slurm.yaml,0.0001,16,0.01,unidirectional,False,True,False,scale_up_3,3,15,big.yaml,0.500522,0.08245858,0.499498,0.08123203
3.35,45,std_runs_slurm.yaml,0.0001,16,0.01,unidirectional,False,True,False,scale_up_3,3,15,big.yaml,0.499497,0.08233703,0.495624,0.08116574
4.35,42,std_runs_slurm.yaml,0.0001,16,0.01,unidirectional,False,True,False,scale_up_3,3,15,big.yaml,0.499880,0.0823221,0.499483,0.08110852
2.36,44,std_runs_slurm.yaml,0.0001,16,0.01,unidirectional,False,True,False,scale_up_3,3,15,big.yaml,0.498147,0.08238765,0.500214,0.08122768


In [37]:
data[(data["model"] == "small.yaml") & (data["model"] == "small.yaml")]["model.model_type"].unique()

array(['convM', 'bidirectional', 'mlp', 'transformer', 'unidirectional'],
      dtype=object)

In [62]:
sum(data["model.corr_regularization"]  == "False") / len(data)

0.8901098901098901

In [63]:
data[data["model.corr_regularization"] == "True"]

HP,seed,hparams_search,model.optimizer_lr,data.batch_size,model.weight_decay,model.model_type,model.corr_regularization,corr_input,regression_head,data.ds_name,max_lags,n_vars,model,Test1_Auroc,Test1_MSE,Test2_AUROC,Test2_MSE
1.6,44,std_runs_slurm.yaml,0.001,64,0,unidirectional,True,True,False,big_base_mirror,3,5,medium.yaml,0.582624,0.08174278,0.613677,0.07955292
3.6,43,std_runs_slurm.yaml,0.001,64,0,unidirectional,True,True,False,big_base_mirror,3,5,medium.yaml,0.498477,0.08323298,0.508215,0.081924126
4.6,42,std_runs_slurm.yaml,0.001,64,0,unidirectional,True,True,False,big_base_mirror,3,5,medium.yaml,0.953328,0.035718296,0.972686,0.02683389
2.6,45,std_runs_slurm.yaml,0.001,64,0,unidirectional,True,True,False,big_base_mirror,3,5,medium.yaml,0.502746,0.083173774,0.511138,0.081866644
0.6,46,std_runs_slurm.yaml,0.001,64,0,unidirectional,True,True,False,big_base_mirror,3,5,medium.yaml,0.804916,0.06554522,0.829007,0.06134754
1.12,42,std_runs_slurm.yaml,0.001,16,0.01,bidirectional,True,True,False,scale_up_2,3,7,big.yaml,0.661042,0.079061225,0.686929,0.077439204
3.12,46,std_runs_slurm.yaml,0.001,16,0.01,bidirectional,True,True,False,scale_up_2,3,7,big.yaml,0.509117,0.08188814,0.512930,0.08187553
4.12,43,std_runs_slurm.yaml,0.001,16,0.01,bidirectional,True,True,False,scale_up_2,3,7,big.yaml,0.499709,0.08187036,0.509249,0.0817977
2.12,45,std_runs_slurm.yaml,0.001,16,0.01,bidirectional,True,True,False,scale_up_2,3,7,big.yaml,0.886865,0.047694236,0.925546,0.03976439
0.12,44,std_runs_slurm.yaml,0.001,16,0.01,bidirectional,True,True,False,scale_up_2,3,7,big.yaml,0.881049,0.049176272,0.927167,0.04042233


In [64]:
res = display_std_runs(data, what = "maxi")
res.to_csv("scores/slurm_exp3_max.csv")
res = display_std_runs(data, what = "mean")
res.to_csv("scores/slurm_exp3_mean.csv")
res = display_std_runs(data, what = "std")
res.to_csv("scores/slurm_exp3_std.csv")

In [38]:
data[["Test1_Auroc", "Test2_AUROC","model.model_type", "data.ds_name" ,  "model"]].groupby(["model.model_type", "data.ds_name",  "model"]).max()

HP                                            Test1_Auroc  Test2_AUROC
model.model_type data.ds_name    model                                
bidirectional    big_base_mirror big.yaml        0.959944     0.986634
                                 medium.yaml     0.943063     0.968218
                                 small.yaml      0.672553     0.704246
                 scale_up        big.yaml        0.499056     0.505268
                                 medium.yaml     0.500371     0.505596
                 scale_up_2      big.yaml        0.886865     0.927167
                                 medium.yaml     0.725411     0.754195
                                 small.yaml      0.507321     0.506244
                 scale_up_3      big.yaml        0.505506     0.501194
convM            big_base_mirror big.yaml        0.509382     0.514226
                                 medium.yaml     0.514772     0.520369
                                 small.yaml      0.518296     0.527030
                 scale_up        big.yaml        0.502303     0.506843
                                 medium.yaml     0.503209     0.506382
                 scale_up_2      big.yaml        0.506019     0.521372
                                 medium.yaml     0.503194     0.511082
                 scale_up_3      big.yaml        0.502086     0.505442
mlp              big_base_mirror medium.yaml     0.504633     0.505843
                                 small.yaml      0.506788     0.506399
                 scale_up        medium.yaml     0.502104     0.505292
                 scale_up_2      medium.yaml     0.504378     0.503824
transformer      big_base_mirror big.yaml        0.992196     0.994416
                                 medium.yaml     0.869706     0.905154
                                 small.yaml      0.684097     0.719196
                 scale_up        big.yaml        0.500747     0.505506
                                 medium.yaml     0.500970     0.503732
                 scale_up_2      big.yaml        0.842551     0.850480
                 scale_up_3      big.yaml        0.501266     0.498701
unidirectional   big_base_mirror big.yaml        0.977436     0.990950
                                 medium.yaml     0.953328     0.972686
                                 small.yaml      0.691948     0.732150
                 scale_up        big.yaml        0.504500     0.504996
                                 medium.yaml     0.500855     0.505295
                 scale_up_2      big.yaml        0.921519     0.936330
                                 medium.yaml     0.719712     0.747751
                                 small.yaml      0.506276     0.509576
                 scale_up_3      big.yaml        0.500951     0.500214

# Experiment 4  - Kuramoto - Grid search

In [41]:
mypath = "/home/stein_test/Causal_pretraining/multirun/experiment_4_grid_kuramoto/"
onlyfiles = [mypath + f for f in listdir(mypath) if "summary" in f]

In [42]:
res = pd.concat([pd.read_csv(x, index_col=0) for x in onlyfiles],axis=1).T

In [43]:
mypath = "/home/stein_test/Causal_pretraining/multirun/experiment_4_std_kuramoto/"


# Std

In [44]:
data = pd.read_csv("multirun/experiment_4_std_kuramoto/summary.csv", index_col=0).T
res = display_std_runs(data, test2=False)

In [45]:
sum(data["model.corr_regularization"]  == "False") / len(data)

1.0

In [46]:
res = display_std_runs(data, test2=False)

In [47]:
res = display_std_runs(data, test2=False, what = "maxi")
res.to_csv("scores/slurm_kura_max.csv")
res = display_std_runs(data, test2=False, what = "mean")
res.to_csv("scores/slurm_kura_mean.csv")
res = display_std_runs(data, test2=False, what = "std")
res.to_csv("scores/slurm_kura_std.csv")